# 01 Data quality

Before any analysis I want to know how much of the data I can trust.
Data: contract award notices from TED, Polish buyers, Jan 2024 - Oct 2026.

In [1]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

con = duckdb.connect("../data/tenders.duckdb", read_only=True)
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)


def q(sql):
    return con.sql(sql).df()

## Notices and lots

One notice can have many lots (parts of the tender). Analysis is done on lot level, because
the number of offers, the winner and the price are reported per lot.

In [2]:
q('''
SELECT
    count(*) AS notices,
    sum(n_lots)::BIGINT AS lots_declared,
    round(avg(lots_parsed::int), 3) AS share_split_into_lots
FROM raw_notices
''')

,notices,lots_declared,share_split_into_lots
0,128133,527341,0.73


The search API returns every field as a flat list. For notices with many lots the lists are not
always in the same order (for example winner ids are deduplicated), so I only split a notice into
lots when the lists clearly match. That works for about 73% of notices. The rest is skipped
on lot level.

In [3]:
q('''
SELECT n_lots > 1 AS multi_lot, lots_parsed, count(*) AS notices
FROM raw_notices
GROUP BY ALL
ORDER BY ALL
''')

,multi_lot,lots_parsed,notices
0,False,False,4076
1,False,True,69956
2,True,False,30984
3,True,True,23117


## Contract values

Values are entered by hand by the buyers and some of them are clearly wrong.

In [4]:
q('''
SELECT value_flag, count(*) AS lots, round(100 * count(*) / sum(count(*)) OVER (), 1) AS pct
FROM fact_lot
GROUP BY 1
ORDER BY 2 DESC
''')

,value_flag,lots,pct
0,ok,187195,90.10
1,missing,11298,5.40
2,above_notice_total,4752,2.30
3,too_low,4396,2.10
4,over_1bn_unverified,134,0.10


Examples of values that were removed. A coaching company winning almost 10 bln PLN or portable
toilets for 9 bln PLN are typos (probably value in grosz or extra zeros). Rules used:

- `too_low` - below 1000 PLN
- `above_notice_total` - lot value higher than the value of the whole notice
- `over_1bn_unverified` - above 1 bln PLN and not confirmed by the buyer's estimate (0.3x - 3x)

In [5]:
q('''
SELECT l.value_flag, d.division_name_en AS category, left(r.winner_name, 45) AS winner,
       round(r.value / 1e6) AS value_mln, round(n.total_value / 1e6) AS notice_total_mln
FROM fact_lot l
JOIN raw_lots r ON r.publication_number = l.publication_number AND r.lot_id = l.lot_id
JOIN raw_notices n ON n.publication_number = l.publication_number
JOIN dim_cpv d USING (cpv_division)
WHERE l.value_flag IN ('above_notice_total', 'over_1bn_unverified')
ORDER BY r.value DESC
LIMIT 10
''')

,value_flag,category,winner,value_mln,notice_total_mln
0,above_notice_total,Financial and insurance services,NaN,"2,532,420.00",7.00
1,over_1bn_unverified,"Petroleum products, fuel, electricity and othe...",ORLEN S.A.,"1,086,150.00","1,086,150.00"
2,above_notice_total,"Petroleum products, fuel, electricity and othe...","ENERGA - OBRÓT S.A.,","87,807.00","21,026.00"
3,over_1bn_unverified,Construction work,Bergerat Monnoyeur Sp. z o.o.,"46,980.00","46,980.00"
4,over_1bn_unverified,Construction work,Przedsiębiorstwo Budowlano-Usługowe Budopol-P,"46,927.00","46,927.00"
5,over_1bn_unverified,Construction work,NaN,"36,878.00","36,878.00"
6,over_1bn_unverified,Repair and maintenance services,ALSTOM Polska S.A,"30,877.00","30,877.00"
7,over_1bn_unverified,Transport equipment and auxiliary products to ...,Solaris Bus & Coach Sp. z o.o. - zadanie 2,"30,580.00","42,688.00"
8,above_notice_total,Construction work,NaN,"27,204.00","22,142.00"
9,over_1bn_unverified,Construction work,URBIT J. MICHALIK SPÓŁKA KOMANDYTOWA,"26,653.00","26,653.00"


Without these rules the 180 lots above 1 bln PLN would make almost half of the whole market value.
After cleaning the biggest contracts look real (trains, postal services, turbines):

In [6]:
q('''
SELECT d.division_name_en AS category, left(winner_name, 40) AS winner,
       round(value_pln / 1e6) AS value_mln, round(estimated_pln / 1e6) AS estimate_mln
FROM fact_lot
JOIN dim_cpv d USING (cpv_division)
WHERE value_pln IS NOT NULL
ORDER BY value_pln DESC
LIMIT 8
''')

,category,winner,value_mln,estimate_mln
0,Transport equipment and auxiliary products to ...,Stadler Polska Sp. z o.o.,"2,642.00","1,976.00"
1,Transport equipment and auxiliary products to ...,NaN,"1,568.00","1,496.00"
2,Postal and telecommunications services,Poczta Polska S.A.,"1,542.00","2,124.00"
3,Postal and telecommunications services,Poczta Polska S.A.,"1,537.00","2,124.00"
4,Transport equipment and auxiliary products to ...,Stadler Polska Sp. z o.o.,"1,329.00","1,021.00"
5,Industrial machinery,Ansaldo Energia S.p.A.,"1,268.00","1,298.00"
6,"Agricultural, forestry, horticultural, aquacul...",Tereny Zielone Marianna Kędziora Marek K,"1,127.00",983.00
7,"Agricultural, forestry, horticultural, aquacul...",Witand Firma Handlowo- Usługowa Andrzej,"1,000.00","1,070.00"


## Number of offers

The count comes from two kinds of statistics: `tenders` (all offers) or `t-esubm` (electronic offers).
In Poland submission is electronic, so I treat both as the number of offers.
Lots with 0, negative or absurd counts (one lot has 201872 offers) are set to missing in `fact_lot`.

In [7]:
q('''
SELECT coalesce(offers_type, 'missing') AS offers_type, count(*) AS lots,
       count(*) FILTER (WHERE n_offers < 1) AS invalid
FROM raw_lots
GROUP BY 1
''')

,offers_type,lots,invalid
0,missing,23596,0
1,t-esubm,109552,1675
2,tenders,74627,1475


## Estimated value

The buyer's estimate is published only for some lots, so the price vs estimate analysis
(notebook 04) uses a smaller sample.

In [8]:
q('''
SELECT
    count(*) AS lots,
    count(estimated_pln) AS with_estimate,
    count(price_to_estimate) AS usable_for_price_analysis,
    round(count(price_to_estimate) / count(*), 3) AS share
FROM fact_lot
''')

,lots,with_estimate,usable_for_price_analysis,share
0,207775,37120,31118,0.15


## Summary

- 128k notices, 208k lot results, 73% of notices split into lots
- about 4.5% of lots have a value that can't be trusted and it is excluded from sums
- number of offers is known for 89% of lots
- price vs estimate can be checked for about 31k lots